# 🚀 SahamBot — Colab Improve Analysis (Eksperimen Baru 2026-11-25)

Notebook ini **khusus untuk improve analisis** berdasarkan hasil audit terbaru:

| Temuan audit | Eksperimen di notebook ini |
|---|---|
| 2/11 sinyal BELI live (PWON, BTPS) ternyata masih **death cross** | **A**: Uji filter death cross (MA50<MA200) pada OOS — apakah menaikkan alpha & menurunkan MDD? |
| Multi-horizon IC: score **60d = +0,024** vs 20d = +0,011 | **B**: Latih model ridge terpisah untuk horizon 60d — apakah lebih baik dari 20d? |
| `anti_rsi` kuat di 5–10d (+0,019/+0,013), negatif di 60d | **C**: Blend anti-RSI (5d) + trend model (60d) — apakah improve risk-adjusted return? |
| Regime `sma50_200_half` sudah dipakai (DSR 0,86) | **D**: Tambah vol-target regime di atas sma50_200_half — uji DSR baru |

**⚠️ Gate promosi (wajib semua lolos sebelum ubah kode produksi):**
1. `ic_model_mean > ic_eq_mean` dan menang di mayoritas tahun OOS
2. `t_hac >= 2.0` (Newey-West, nlags=horizon)
3. `alpha_net CI95 lower bound > 0` setelah fee BEI 0,40%
4. `DSR >= 0.95` dengan `n_trials` yang jujur
5. `tools/check_sync.py` PASS + `/jejak` live konsisten

> Bobot `core/relatif.py` **tetap beku** sampai semua gate lolos. Notebook ini hanya menghasilkan **rekomendasi + data**, bukan langsung mengubah bot.


In [ ]:
#@title 0. Setup — Clone repo, install deps, panel konfigurasi
import os, pathlib

FAST_MODE = True              #@param {type:"boolean"}  # True = cepat; False = audit penuh
USE_DRIVE_CACHE = True        #@param {type:"boolean"}  # True = cache data 10thn di Google Drive
RUN_EXP_A_DEATHCROSS = True   #@param {type:"boolean"}  # Eksperimen A: filter death cross
RUN_EXP_B_H60 = True          #@param {type:"boolean"}  # Eksperimen B: model horizon 60d
RUN_EXP_C_BLEND = True        #@param {type:"boolean"}  # Eksperimen C: blend anti-RSI 5d + trend
RUN_EXP_D_VOLTARGET = True    #@param {type:"boolean"}  # Eksperimen D: vol-target regime
PUBLISH = False               #@param {type:"boolean"}  # True = tulis hasil ke data/validation.json (HATI-HATI!)
PUSH_TO_GITHUB = True         #@param {type:"boolean"}  # True = commit & push hasil ke repo

REPO = "https://github.com/bagasadi05/saham-bot.git"
BRANCH = "main"
if not pathlib.Path("saham-bot").exists():
    !git clone --depth 1 -b {BRANCH} {REPO}
%cd saham-bot
!git pull --ff-only 2>/dev/null || true
!pip install -q -r requirements.txt lightgbm

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["COLAB_FAST_MODE"] = "1" if FAST_MODE else "0"

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    drv = pathlib.Path("/content/drive/MyDrive/sahambot_research_cache")
    drv.mkdir(parents=True, exist_ok=True)
    local = pathlib.Path("temp/research/prices")
    local.parent.mkdir(parents=True, exist_ok=True)
    if not local.exists():
        local.symlink_to(drv)
    print(f"[OK] Drive cache: {drv}")

print(f"[OK] Setup selesai. FAST={FAST_MODE} A={RUN_EXP_A_DEATHCROSS} B={RUN_EXP_B_H60} C={RUN_EXP_C_BLEND} D={RUN_EXP_D_VOLTARGET}")


## 1. Baseline — Evaluasi walk-forward yang sudah ada

Dapatkan angka baseline saat ini (`ic_model_mean`, DSR, alpha net, MDD) sebagai pembanding eksperimen baru.


In [ ]:
#@title 1a. Walk-forward baseline (20d)
!python research/evaluate.py --period 10y --horizon 20


In [ ]:
#@title 1b. Multi-horizon IC (5/10/20/60d) — konfirmasi temuan audit
!python research/horizons.py --period 10y


In [ ]:
#@title 1c. Portfolio baseline + DSR + ablasi eksperimen
!python research/portfolio.py --hold 0.6 --min-hold 2 --regime sma50_200_half --offline
!python research/experiments.py --offline --horizons 20 40 60


## 2. Eksperimen A — Death Cross Filter

**Hipotesis:** Menyaring saham dengan `MA50 < MA200` (death cross) dari kandidat BELI akan menurunkan MDD dan bisa menaikkan net alpha.

**Metode:** Pakai OOS predictions (`temp/research/oos_predictions.pkl`), hitung death cross per (date, ticker) dari harga, lalu jalankan portfolio dengan & tanpa filter.


In [ ]:
#@title 2a. Eksperimen A: build death-cross-filtered OOS predictions
import os
import pandas as pd

SRC = "temp/research/oos_predictions.pkl"
DS = "temp/research/dataset_10y_45.pkl"

if RUN_EXP_A_DEATHCROSS and os.path.exists(SRC) and os.path.exists(DS):
    preds = pd.read_pickle(SRC)
    raw = pd.read_pickle(DS)
    print(f"OOS predictions: {len(preds)} baris, kolom: {list(preds.columns)[:10]}")

    # Bangun price matrix date x ticker
    if isinstance(raw.index, pd.MultiIndex):
        px = raw["close"].unstack(level=-1)
    else:
        px = raw["close"] if "close" in raw else raw
    sma50 = px.rolling(50).mean()
    sma200 = px.rolling(200).mean()
    death_cross = (sma50 < sma200)

    # Tempel flag ke predictions
    if "ticker" in preds.columns and "date" in preds.columns:
        dc_map = death_cross.stack()
        preds["death_cross"] = [
            bool(dc_map.get((d, t), False))
            for d, t in zip(preds["date"], preds["ticker"], strict=False)
        ]
        n_dc = int(preds["death_cross"].sum())
        print(f"Death cross di OOS: {n_dc}/{len(preds)} ({n_dc/len(preds)*100:.1f}%)")

        # Varian filtered: prediksi saham death cross di-set ke median (netral)
        pf = preds.copy()
        if "pred" in pf.columns:
            med = pf["pred"].median()
            pf.loc[pf["death_cross"], "pred"] = med
        pf.to_pickle("temp/research/oos_predictions_no_dc.pkl")
        print("[OK] Tersimpan: temp/research/oos_predictions_no_dc.pkl")
else:
    print("[SKIP] Jalankan section 1 dulu agar file OOS predictions tersedia.")


In [ ]:
#@title 2b. Eksperimen A: portfolio dengan & tanpa death cross
if RUN_EXP_A_DEATHCROSS:
    print("=== BASELINE (tanpa filter death cross) ===")
    !python research/portfolio.py --hold 0.6 --min-hold 2 --regime sma50_200_half --offline --src temp/research/oos_predictions.pkl
    print("\n=== FILTERED (tanpa death cross) ===")
    !python research/portfolio.py --hold 0.6 --min-hold 2 --regime sma50_200_half --offline --src temp/research/oos_predictions_no_dc.pkl
else:
    print("[SKIP] Eksperimen A dimatikan.")


## 3. Eksperimen B — Model Horizon 60d

**Hipotesis:** IC score di 60d (+0,024) lebih kuat dari 20d (+0,011). Latih ridge walk-forward dengan `--horizon 60` dan bandingkan.

**Hati-hati:** horizon lebih panjang = sinyal lebih lambat, cocok untuk swing trader.


In [ ]:
#@title 3a. Walk-forward horizon 60d
if RUN_EXP_B_H60:
    !python research/evaluate.py --period 10y --horizon 60 --out temp/research/h60
else:
    print("[SKIP] Eksperimen B dimatikan.")


In [ ]:
#@title 3b. Portfolio horizon 60d (bandingkan DSR & MDD)
if RUN_EXP_B_H60:
    !python research/portfolio.py --hold 0.6 --min-hold 2 --regime sma50_200_half --step 60 --offline --src temp/research/h60/oos_predictions.pkl 2>/dev/null || echo "Jalankan 3a dulu untuk dapat OOS 60d"
else:
    print("[SKIP]")


## 4. Eksperimen C — Blend Anti-RSI (5d) + Trend Model

**Hipotesis:** `anti_rsi` IC positif di 5d (+0,019), trend model kuat di 60d. Blend keduanya bisa lebih baik di berbagai kondisi.

Uji berbagai bobot `w` (0, 0.25, 0.5, 0.75, 1.0) — semua dihitung dalam `n_trials` DSR.


In [ ]:
#@title 4a. Blend anti-RSI proxy (5d) + trend model, berbagai bobot
if RUN_EXP_C_BLEND and os.path.exists(DS) and os.path.exists(SRC):
    import numpy as np
    import pandas as pd

    raw = pd.read_pickle(DS)
    px = raw["close"].unstack(level=-1) if isinstance(raw.index, pd.MultiIndex) else raw["close"]
    ret5 = px.pct_change(5)
    # anti-RSI proxy: return 5d negatif = oversold = beli (z-score cross-sectional)
    anti = -ret5.sub(ret5.mean(axis=1), axis=0).div(ret5.std(axis=1), axis=0)

    preds = pd.read_pickle(SRC)
    if "pred" in preds.columns:
        trend = preds.pivot(index="date", columns="ticker", values="pred")
        trend_z = trend.sub(trend.mean(axis=1), axis=0).div(trend.std(axis=1), axis=0)
        common = anti.index.intersection(trend_z.index)
        fwd20 = px.pct_change(20).shift(-20)
        print("w_anti | IC_blend_20d | n_dates")
        for w in [0.0, 0.25, 0.5, 0.75, 1.0]:
            blend = (1 - w) * trend_z.loc[common] + w * anti.loc[common]
            ics = []
            for d in common:
                if d in fwd20.index:
                    x = blend.loc[d].dropna()
                    y = fwd20.loc[d, x.index].dropna()
                    if len(x) > 10:
                        ics.append(x.corr(y))
            if ics:
                print(f"{w:>8.2f} | {np.mean(ics):>+12.4f} | {len(ics)}")
else:
    print("[SKIP] Eksperimen C dimatikan atau dataset belum tersedia.")


## 5. Eksperimen D — Vol-Target Regime

**Hipotesis:** `vol_target_14` (target vol 14%) di atas `sma50_200_half` bisa menurunkan MDD saat pasar volatile.


In [ ]:
#@title 5a. Bandingkan regime: none vs sma50_200_half vs vol_target_14
if RUN_EXP_D_VOLTARGET:
    for reg in ["none", "sma50_200_half", "vol_target_14"]:
        print(f"\n=== regime={reg} ===")
        !python research/portfolio.py --hold 0.6 --min-hold 2 --regime {reg} --offline
else:
    print("[SKIP] Eksperimen D dimatikan.")


## 6. Ringkasan & Rekomendasi

| Eksperimen | Metric | Baseline | Varian Baru | Gate Lolos? |
|---|---|---|---|---|
| A: Death cross | alpha_net / MDD / DSR | section 1c | section 2b | ☐ |
| B: Horizon 60d | ic_model / t_hac / DSR | 20d: 0,0304/0,85/0,86 | section 3 | ☐ |
| C: Blend | IC blend / DSR | 0,0304 | section 4 | ☐ |
| D: Vol-target | MDD / DSR | -20,0%/0,86 | section 5 | ☐ |

**Aturan:** rekomendasi baru hanya di-endorse jika semua gate lolos (DSR ≥ 0.95, t_hac ≥ 2.0, CI95 lo > 0).


In [ ]:
#@title 6a. Ringkasan otomatis hasil
import glob, json, os

print("=" * 60)
print("RINGKASAN HASIL EKSPERIMEN")
print("=" * 60)
if os.path.exists("data/validation.json"):
    v = json.load(open("data/validation.json"))
    print(f"[BASELINE] keys: {list(v.keys())}")
for f in sorted(glob.glob("temp/research/*.md")):
    print(f"\n--- {f} (500 char pertama) ---")
    print(open(f).read()[:500])


## 7. Publish (OPSIONAL — hanya jika semua gate lolos!)

⚠️ Jangan jalankan kecuali: DSR ≥ 0.95, check_sync PASS, sudah review manual.


In [ ]:
#@title 7a. Publish hasil
if PUBLISH:
    !python research/publish.py
    !python tools/check_sync.py
else:
    print("[SKIP] PUBLISH=False — hasil lokal saja, tidak mengubah validation.json")


## 8. Commit & Push ke GitHub (opsional)


In [ ]:
#@title 8a. Commit & push hasil riset
if PUSH_TO_GITHUB:
    import getpass
    token = getpass.getpass("GitHub PAT untuk push: ")
    !git config user.email "colab@sahambot.local"
    !git config user.name "SahamBot Colab"
    !git add research/colab_improve_analysis.ipynb
    !git add data/validation.json 2>/dev/null || true
    !git commit -m "chore(research): update hasil eksperimen improve analysis dari Colab" || echo "Tidak ada perubahan"
    remote = f"https://{token}@github.com/bagasadi05/saham-bot.git"
    !git push {remote} HEAD:main 2>&1 || echo "[WARN] Push gagal — cek token"
else:
    print("[SKIP] PUSH_TO_GITHUB=False")


## 9. Langkah setelah Colab selesai

1. `git pull origin main` di lokal
2. Verifikasi: `python tools/check_sync.py` + `pytest -q` + `ruff check .`
3. Review manual hasil di `temp/research/evaluation.md` & `portfolio.md`
4. Kalau gate lolos → update kode produksi + tes + commit terpisah
5. Kalau tidak lolos → bobot tetap beku, catat di CHANGELOG

> Prinsip: angka riset hanya dari perintah yang dijalankan.
